# Experiment 026a — Granularity Screening

Low-cost mechanism screen before the longer Experiment 026 confirmation. G={1,4,8}, 5M continuation tokens per arm, 15M total training tokens. Target: Tesla T4×2.


In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/experiment-026a-granularity-screening'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'
if not (ROOT / '.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(ROOT)], check=True)
else:
    subprocess.run(['git','fetch','origin',BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git','checkout',BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git','reset','--hard',f'origin/{BRANCH}'], cwd=ROOT, check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.[dev]'], cwd=ROOT, check=True)
print(subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip())


In [ ]:
subprocess.run([sys.executable,'-m','pytest','tests/research/02-self-organization/test_developmental_tissue.py','tests/research/03-routing-and-growth/test_experiment_026_cell_granularity.py','tests/research/03-routing-and-growth/test_experiment_026a_screening.py','-q'], cwd=ROOT, check=True)
subprocess.run([sys.executable,'scripts/research/run_experiment_026_cell_granularity_smoke.py'], cwd=ROOT, check=True)


In [ ]:
import torch
print({'gpu_count': torch.cuda.device_count(), 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
assert torch.cuda.device_count() >= 2, 'Experiment 026a expects T4×2'


In [ ]:
subprocess.run([sys.executable,'scripts/research/run_experiment_026a_granularity_screening.py'], cwd=ROOT, check=True)


In [ ]:
import json
from IPython.display import Image, display
OUT = ROOT / 'results' / 'experiment-026a-granularity-screening'
summary = json.loads((OUT/'worker-summary.json').read_text())
print(json.dumps(summary, indent=2))
decision = json.loads((OUT/'decision.json').read_text()) if (OUT/'decision.json').is_file() else {}
print(json.dumps(decision, indent=2))
plot = OUT / 'screening-specialization.png'
if plot.is_file(): display(Image(filename=str(plot)))
